# 🔮 FORESIGHT — 05: Machine Learning & Gradient Boosting Forecasters

This notebook explores machine learning regression and ensemble trees:
- **Ridge Linear Regression** (Standardized linear baseline)
- **Random Forest Regressor** (Bagging ensemble)
- **Histogram Gradient Boosting (HistGBM)**
- **Extreme Gradient Boosting (XGBoost)**
- **Quantile Gradient Boosting** ($P_{10}, P_{50}, P_{90}$)

In [ ]:
import pandas as pd
import numpy as np
from foresight.features.pipeline import FeatureEngineeringPipeline
from foresight.forecasting.ml_models import (
    LinearRegressionForecaster,
    RandomForestForecaster,
    GradientBoostingForecaster,
    XGBoostForecaster,
    QuantileGradientBoostingForecaster,
)
from foresight.evaluation.metrics import evaluate_predictions

df = pd.read_parquet('data/processed/features_engineered.parquet')
pipeline = FeatureEngineeringPipeline()
feature_names = pipeline.get_feature_names(df)

dates = pd.to_datetime(df["date"])
val_start = dates.max() - pd.Timedelta(days=30)

train_df = df[dates < val_start]
val_df = df[dates >= val_start]

X_train, y_train = train_df[feature_names], train_df["quantity"].values
X_val, y_val = val_df[feature_names], val_df["quantity"].values
print(f"Training set: {X_train.shape} | Validation set: {X_val.shape}")

## 2. Fit ML Model Suite & Compare Against Baselines

In [ ]:
ml_models = [
    LinearRegressionForecaster(),
    RandomForestForecaster(n_estimators=50, max_depth=10),
    GradientBoostingForecaster(max_iter=80, max_depth=6),
    XGBoostForecaster(n_estimators=100, max_depth=6, learning_rate=0.08),
]

print("=== MACHINE LEARNING MODEL LEADERBOARD ===")
print(f"{'Model Name':<35} | {'WAPE':<8} | {'RMSE':<8} | {'MAE':<8} | {'R²':<8}")
print("-" * 75)

for model in ml_models:
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    scores = evaluate_predictions(y_val, preds, y_train=y_train)
    print(f"{model.name:<35} | {scores.wape:<8.4f} | {scores.rmse:<8.2f} | {scores.mae:<8.2f} | {scores.r2:<8.3f}")

## 3. Probabilistic Quantile Forecast ($P_{10}, P_{50}, P_{90}$)

In [ ]:
q_forecaster = QuantileGradientBoostingForecaster(quantiles=[0.10, 0.50, 0.90], max_iter=60)
q_forecaster.fit(X_train, y_train)
quantiles_pred = q_forecaster.predict_quantiles(X_val)

p10 = quantiles_pred[0.10]
p50 = quantiles_pred[0.50]
p90 = quantiles_pred[0.90]

coverage_80 = np.mean((y_val >= p10) & (y_val <= p90)) * 100
print(f"Empirical 80% Prediction Interval Coverage [P10 to P90]: {coverage_80:.2f}%")